# Cost Attribution on Databricks — Walkthrough

**Goal:** Attribute Databricks DBU cost three ways using GA system tables.

| View | Question it answers | Source tables |
|------|---------------------|---------------|
| 👤 Per-user | "Which user / service principal is driving cost?" | `system.billing.usage` + `system.billing.list_prices` |
| 🔍 Per-query | "Approximately how much did each SQL query cost?" | `system.query.history` × `system.billing.usage` |
| 📦 Per-table / view / MV | "Which UC table or MV is most expensive to maintain?" | `system.billing.usage.usage_metadata.uc_table_*` |

The companion AI/BI dashboard uses these exact queries.

## Step 0 — The system tables you'll use

| Table | What's in it | Grain |
|-------|--------------|-------|
| `system.billing.usage` | Every billable DBU row from every workload | One row per (workspace, sku, hour, compute resource) |
| `system.billing.list_prices` | Time-versioned $ price per SKU per cloud | One row per (sku, cloud, validity window) |
| `system.query.history` | Every SQL statement executed on a warehouse | One row per `statement_id` |

The `usage_metadata` and `identity_metadata` structs on `system.billing.usage` are the **dimensions** we use to attribute cost. Inspect them first.

In [0]:
DESCRIBE system.billing.usage;

In [0]:
-- The two structs that drive attribution
SELECT
  identity_metadata.run_as       AS run_as,         -- email or SP for who/what ran the workload
  identity_metadata.created_by   AS created_by,     -- creator of the resource (cluster/warehouse/job)
  identity_metadata.owned_by     AS owned_by,       -- owner of the resource
  usage_metadata.warehouse_id    AS warehouse_id,   -- DBSQL warehouse that ran the query
  usage_metadata.cluster_id      AS cluster_id,     -- All-purpose / job cluster
  usage_metadata.job_id          AS job_id,         -- Workflow job
  usage_metadata.dlt_pipeline_id AS dlt_pipeline_id,-- DLT / streaming
  usage_metadata.uc_table_catalog AS uc_catalog,
  usage_metadata.uc_table_schema  AS uc_schema,
  usage_metadata.uc_table_name    AS uc_table_name -- UC table / view / MV the DBUs hit
FROM system.billing.usage
WHERE usage_date >= current_date() - 7
LIMIT 5;

## Step 1 — Convert raw DBUs into dollars: the `list_prices` join

`system.billing.usage.usage_quantity` is in **DBUs** (or hours, or GB, depending on the SKU). To translate to dollars we look up the unit price in `system.billing.list_prices` — but with two important rules:

1. **Join on `sku_name + usage_unit`**, not just `sku_name + cloud`. A single SKU can have multiple unit types (DBU, hour, GB) and we need to pick the right row.
2. **Bind to the price valid at the time of usage**, not today's price. If list prices change mid-window, this keeps history accurate.

We use `pricing.effective_list.default` for the unit price — that's the same field the standard *Account Usage Dashboard V2* uses, so the totals reconcile exactly.

In [ ]:
-- The reusable price CTE — every downstream query joins on this.
-- We bind to the price that was valid at the time of the usage (not today's
-- price), and join on sku_name + usage_unit to avoid fanning out on SKUs
-- that have multiple unit types.
WITH prices AS (
  SELECT
    sku_name,
    usage_unit,
    price_start_time,
    COALESCE(price_end_time, date_add(current_date, 1)) AS price_end_time,
    pricing.effective_list.default AS unit_price            -- $ per usage_unit (DBU, hour, GB, ...)
  FROM system.billing.list_prices
  WHERE currency_code = 'USD'
)
SELECT * FROM prices ORDER BY sku_name LIMIT 10;


## Step 2 — 👤 Per-user attribution

**The naive approach (and why it fails for warehouses).** A first try is to group `system.billing.usage` by `identity_metadata.run_as`. That works for jobs, all-purpose clusters, and DLT pipelines. But it fails for **DBSQL warehouses**: a warehouse is a shared resource, so `system.billing.usage` charges it by **warehouse uptime**, not per query. The billing row has no `executed_by_user_id` — it just says "warehouse X consumed Y DBUs on day Z." Every warehouse-driven row would fall into `(unattributed)`.

**The right approach.** Use `system.query.history.total_duration_ms` (which *does* have `executed_by`) to allocate each warehouse-day's cost across the users that ran queries that day, proportional to runtime. For everything that isn't a warehouse — jobs, clusters, DLT, etc. — fall back to `identity_metadata.run_as` as before.

The dataset is a UNION of three sources:

1. **Warehouse cost, allocated by query runtime.** Per `(workspace, warehouse, day)`, total cost from `billing.usage` is split across users by `SUM(total_duration_ms)` share from `query.history`.
2. **Warehouse idle uptime.** Any warehouse-day with no queries in `query.history` is bucketed as `(unattributed — warehouse idle)` so totals still reconcile.
3. **Non-warehouse rows.** Jobs / clusters / DLT / serving / apps / infra → `identity_metadata.run_as`.

In [ ]:
WITH prices AS (
  SELECT sku_name, usage_unit, price_start_time,
         COALESCE(price_end_time, date_add(current_date, 1)) AS price_end_time,
         pricing.effective_list.default AS unit_price
  FROM system.billing.list_prices WHERE currency_code = 'USD'
),
wh AS (
  SELECT warehouse_id, FIRST(warehouse_name IGNORE NULLS) AS warehouse_name
  FROM system.compute.warehouses GROUP BY warehouse_id
),
cl AS (
  SELECT cluster_id, FIRST(cluster_name IGNORE NULLS) AS cluster_name
  FROM system.compute.clusters GROUP BY cluster_id
),
jn AS (
  SELECT job_id, FIRST(name IGNORE NULLS) AS job_name
  FROM system.lakeflow.jobs GROUP BY job_id
),

-- 1) warehouse-day cost from billing.usage
warehouse_day_cost AS (
  SELECT
    u.workspace_id,
    u.usage_metadata.warehouse_id AS warehouse_id,
    u.usage_date,
    u.billing_origin_product AS product,
    SUM(u.usage_quantity * COALESCE(p.unit_price, 0)) AS day_cost_usd,
    SUM(u.usage_quantity)                              AS day_dbus
  FROM system.billing.usage u
  LEFT JOIN prices p
    ON  u.sku_name   = p.sku_name
    AND u.usage_unit = p.usage_unit
    AND u.usage_end_time BETWEEN p.price_start_time AND p.price_end_time
  WHERE u.usage_metadata.warehouse_id IS NOT NULL
    AND u.usage_date >= current_date() - 30
  GROUP BY 1, 2, 3, 4
),

-- 2) per-user runtime share from query.history (this is the new piece)
user_day_runtime AS (
  SELECT
    workspace_id,
    compute.warehouse_id           AS warehouse_id,
    DATE(start_time)               AS usage_date,
    COALESCE(executed_by, '(unattributed)') AS user_email,
    SUM(total_duration_ms)         AS user_ms
  FROM system.query.history
  WHERE compute.warehouse_id IS NOT NULL
    AND start_time >= current_date() - 30
  GROUP BY 1, 2, 3, 4
),
warehouse_day_total_runtime AS (
  SELECT workspace_id, warehouse_id, usage_date,
         SUM(user_ms) AS day_total_ms
  FROM user_day_runtime
  GROUP BY 1, 2, 3
),

-- 3) allocate warehouse cost per user proportional to runtime
warehouse_attributed AS (
  SELECT
    udr.user_email,
    wdc.workspace_id,
    wdc.usage_date,
    wdc.product,
    CONCAT('[Warehouse] ', COALESCE(wh.warehouse_name, wdc.warehouse_id)) AS compute_instance,
    (udr.user_ms * 1.0 / NULLIF(wdtr.day_total_ms, 0)) * wdc.day_cost_usd AS cost_usd,
    (udr.user_ms * 1.0 / NULLIF(wdtr.day_total_ms, 0)) * wdc.day_dbus     AS dbus
  FROM warehouse_day_cost wdc
  JOIN warehouse_day_total_runtime wdtr USING (workspace_id, warehouse_id, usage_date)
  JOIN user_day_runtime udr            USING (workspace_id, warehouse_id, usage_date)
  LEFT JOIN wh ON wh.warehouse_id = wdc.warehouse_id
),

-- 4) warehouse uptime with zero queries that day → keep total whole
warehouse_idle AS (
  SELECT
    '(unattributed — warehouse idle)' AS user_email,
    wdc.workspace_id,
    wdc.usage_date,
    wdc.product,
    CONCAT('[Warehouse] ', COALESCE(wh.warehouse_name, wdc.warehouse_id)) AS compute_instance,
    wdc.day_cost_usd AS cost_usd,
    wdc.day_dbus     AS dbus
  FROM warehouse_day_cost wdc
  LEFT JOIN warehouse_day_total_runtime wdtr USING (workspace_id, warehouse_id, usage_date)
  LEFT JOIN wh ON wh.warehouse_id = wdc.warehouse_id
  WHERE wdtr.day_total_ms IS NULL OR wdtr.day_total_ms = 0
),

-- 5) non-warehouse rows still use identity_metadata.run_as
non_warehouse AS (
  SELECT
    COALESCE(u.identity_metadata.run_as, '(unattributed)') AS user_email,
    u.workspace_id,
    u.usage_date,
    u.billing_origin_product AS product,
    CASE
      WHEN u.usage_metadata.job_id IS NOT NULL
        THEN CONCAT('[Job] ',      COALESCE(jn.job_name,     u.usage_metadata.job_id))
      WHEN u.usage_metadata.cluster_id IS NOT NULL
        THEN CONCAT('[Cluster] ',  COALESCE(cl.cluster_name, u.usage_metadata.cluster_id))
      WHEN u.usage_metadata.dlt_pipeline_id IS NOT NULL
        THEN CONCAT('[Pipeline] ', u.usage_metadata.dlt_pipeline_id)
      WHEN u.usage_metadata.endpoint_id IS NOT NULL OR u.usage_metadata.endpoint_name IS NOT NULL
        THEN CONCAT('[Endpoint] ', COALESCE(u.usage_metadata.endpoint_name, u.usage_metadata.endpoint_id))
      WHEN u.usage_metadata.app_id IS NOT NULL OR u.usage_metadata.app_name IS NOT NULL
        THEN CONCAT('[App] ',      COALESCE(u.usage_metadata.app_name, u.usage_metadata.app_id))
      ELSE '[Other] (none)'
    END AS compute_instance,
    u.usage_quantity * COALESCE(p.unit_price, 0) AS cost_usd,
    u.usage_quantity                              AS dbus
  FROM system.billing.usage u
  LEFT JOIN prices p
    ON  u.sku_name   = p.sku_name
    AND u.usage_unit = p.usage_unit
    AND u.usage_end_time BETWEEN p.price_start_time AND p.price_end_time
  LEFT JOIN cl ON cl.cluster_id = u.usage_metadata.cluster_id
  LEFT JOIN jn ON jn.job_id     = u.usage_metadata.job_id
  WHERE u.usage_metadata.warehouse_id IS NULL
    AND u.usage_date >= current_date() - 30
)

SELECT user_email,
       SUM(cost_usd) AS cost_usd,
       SUM(dbus)     AS dbus
FROM (
  SELECT user_email, CAST(cost_usd AS DOUBLE) AS cost_usd, CAST(dbus AS DOUBLE) AS dbus FROM warehouse_attributed
  UNION ALL
  SELECT user_email, CAST(cost_usd AS DOUBLE), CAST(dbus AS DOUBLE) FROM warehouse_idle
  UNION ALL
  SELECT user_email, CAST(cost_usd AS DOUBLE), CAST(dbus AS DOUBLE) FROM non_warehouse
) all_rows
GROUP BY 1
ORDER BY cost_usd DESC NULLS LAST
LIMIT 30;


### Variations to know

- **Service-principal cost** → `executed_by LIKE '%@servicePrincipal%'` (queries) or `identity_metadata.run_as LIKE '%@servicePrincipal%'` (jobs/clusters).
- **Cap idle uptime fairly** → if `(unattributed — warehouse idle)` is large, the warehouse is staying warm with no queries. Lower `auto_stop_mins` on the warehouse, or split the idle cost across the warehouse's recent active users with another window.
- **Per-product slicing** → group by `product` to see whether the user's cost is SQL warehouse, all-purpose clusters, jobs, etc.

## Step 3 — 🔍 Per-query approximate cost

**Why "approximate"?** DBSQL is billed by **warehouse uptime**, not per-statement. A query running for 5 s on a warm warehouse doesn't cost the same as a 5 s query on a cold one — what you actually pay is *the warehouse-hour*, regardless of how many queries shared it.

**The trick.** Allocate each warehouse-day's $ cost to queries by their **share of total query duration on that warehouse-day**:

```
approx_cost(query) = (total_duration_ms / Σ total_duration_ms on the same warehouse-day)
                     × $ cost of that warehouse on that day
```

This is **directionally correct** for chargeback but loses precision when:

- The warehouse is mostly idle (cost is dominated by uptime, not queries) — a heavy query "wins" the allocation it didn't actually drive.
- Queries run in parallel — proportional duration overestimates a long-running query that shared compute.

**Joins:**
```
query_runtime  qr  (one row per statement, with day-total duration via window fn)
  JOIN warehouse_cost  wc  (one row per workspace × warehouse × day, $ from billing.usage)
  ON workspace_id, warehouse_id, usage_date
```

In [ ]:
WITH prices AS (
  SELECT
    sku_name,
    usage_unit,
    price_start_time,
    COALESCE(price_end_time, date_add(current_date, 1)) AS price_end_time,
    pricing.effective_list.default AS unit_price            -- $ per usage_unit (DBU, hour, GB, ...)
  FROM system.billing.list_prices
  WHERE currency_code = 'USD'
),

-- a) cost per warehouse-day from billing.usage
warehouse_cost AS (
  SELECT
    u.workspace_id,
    u.usage_metadata.warehouse_id                    AS warehouse_id,
    u.usage_date,
    SUM(u.usage_quantity * COALESCE(p.unit_price, 0)) AS day_cost_usd
  FROM system.billing.usage u
  LEFT JOIN prices p
    ON  u.sku_name   = p.sku_name
    AND u.usage_unit = p.usage_unit
    AND u.usage_end_time BETWEEN p.price_start_time AND p.price_end_time
  WHERE u.usage_metadata.warehouse_id IS NOT NULL
    AND u.usage_date >= current_date() - 30
  GROUP BY 1, 2, 3
),

-- b) per-statement runtime + day-total runtime via window function
query_runtime AS (
  SELECT
    workspace_id,
    compute.warehouse_id          AS warehouse_id,
    DATE(start_time)              AS usage_date,
    statement_id,
    executed_by,
    statement_type,
    SUBSTRING(statement_text, 1, 200) AS statement_preview,
    total_duration_ms,
    SUM(total_duration_ms) OVER (
      PARTITION BY workspace_id, compute.warehouse_id, DATE(start_time)
    )                              AS day_total_ms
  FROM system.query.history
  WHERE compute.warehouse_id IS NOT NULL
    AND start_time >= current_date() - 30
)

-- c) allocate each warehouse-day's cost to its queries proportionally
SELECT
  qr.statement_id,
  qr.executed_by                              AS user_email,
  qr.usage_date,
  qr.warehouse_id,
  qr.statement_type,
  qr.statement_preview,
  qr.total_duration_ms,
  (qr.total_duration_ms * 1.0
     / NULLIF(qr.day_total_ms, 0))
   * wc.day_cost_usd                          AS approx_cost_usd
FROM query_runtime qr
JOIN warehouse_cost wc USING (workspace_id, warehouse_id, usage_date)
ORDER BY approx_cost_usd DESC
LIMIT 20;


### Variations to know

- **Allocate by `read_bytes` instead of duration** → swap `total_duration_ms` for `read_bytes` in the window + numerator. Closer to "data-shuffled" cost; biased against compute-heavy queries with small reads.
- **Filter to a dashboard or BI tool** → join on `query_source.dashboard_id` or `client_application` to attribute cost to a Tableau / Power BI / Genie surface.
- **Roll up to query "shape"** → group by `regexp_replace(statement_text, '[0-9]+', 'N')` to find the most expensive *query patterns*, not individual runs.
- **Job / notebook cost** → similar pattern, but join on `usage_metadata.job_id` / `notebook_id` from billing.usage rather than warehouse_id, and use `total_task_duration_ms` from `system.lakeflow.job_run_timeline` if you want job-grain.

## Step 4 — 📦 Per-table / view / materialized-view attribution

**Logic.** `system.billing.usage.usage_metadata.uc_table_catalog/schema/name` is **populated by the platform** for any UC-aware workload — DLT pipelines, streaming tables, materialized views, and (increasingly) SQL workloads tagged to UC tables. Group by these three fields and apply the same price join.

**What's covered:**

- DLT / Lakeflow Pipelines updating a streaming table → cost rolls up to the **target table**.
- Materialized View refresh → cost rolls up to the **MV**.
- SQL warehouse queries against UC tables — partial coverage; many SQL rows still don't carry `uc_table_*`. The query-level allocation in Step 3 fills that gap.

**Joins:**
```
system.billing.usage  u
  ← LEFT JOIN system.billing.list_prices  ON sku_name + cloud
```
No second join needed — `uc_table_*` is already in the same row.

In [ ]:
WITH prices AS (
  SELECT
    sku_name,
    usage_unit,
    price_start_time,
    COALESCE(price_end_time, date_add(current_date, 1)) AS price_end_time,
    pricing.effective_list.default AS unit_price            -- $ per usage_unit (DBU, hour, GB, ...)
  FROM system.billing.list_prices
  WHERE currency_code = 'USD'
)
SELECT
  u.workspace_id,
  u.usage_metadata.uc_table_catalog AS catalog,
  u.usage_metadata.uc_table_schema  AS schema_name,
  u.usage_metadata.uc_table_name    AS table_fqn,
  u.billing_origin_product          AS product,
  u.usage_date,
  SUM(u.usage_quantity * COALESCE(p.unit_price, 0)) AS cost_usd,
  SUM(u.usage_quantity)                             AS dbus
FROM system.billing.usage u
LEFT JOIN prices p
  ON  u.sku_name   = p.sku_name
  AND u.usage_unit = p.usage_unit
  AND u.usage_end_time BETWEEN p.price_start_time AND p.price_end_time
WHERE u.usage_date >= current_date() - 30
  AND u.usage_metadata.uc_table_name IS NOT NULL
GROUP BY 1, 2, 3, 4, 5, 6
ORDER BY cost_usd DESC
LIMIT 20;


### "Cost of an MV refresh" — direct answer
To answer **"What does materialized view `cat.sch.my_mv` cost me per day?"**:

In [ ]:
WITH prices AS (
  SELECT
    sku_name,
    usage_unit,
    price_start_time,
    COALESCE(price_end_time, date_add(current_date, 1)) AS price_end_time,
    pricing.effective_list.default AS unit_price            -- $ per usage_unit (DBU, hour, GB, ...)
  FROM system.billing.list_prices
  WHERE currency_code = 'USD'
)
SELECT
  u.usage_date,
  CONCAT_WS('.',
    u.usage_metadata.uc_table_catalog,
    u.usage_metadata.uc_table_schema,
    u.usage_metadata.uc_table_name)                 AS table_fqn,
  SUM(u.usage_quantity * COALESCE(p.unit_price, 0)) AS cost_usd,
  SUM(u.usage_quantity)                             AS dbus
FROM system.billing.usage u
LEFT JOIN prices p
  ON  u.sku_name   = p.sku_name
  AND u.usage_unit = p.usage_unit
  AND u.usage_end_time BETWEEN p.price_start_time AND p.price_end_time
WHERE u.usage_date >= current_date() - 30
  AND u.usage_metadata.uc_table_catalog IS NOT NULL
  -- Replace these three placeholders with the actual MV you care about:
  -- AND u.usage_metadata.uc_table_catalog = 'main'
  -- AND u.usage_metadata.uc_table_schema  = 'analytics'
  -- AND u.usage_metadata.uc_table_name    = 'daily_revenue_mv'
GROUP BY 1, 2
ORDER BY 1 DESC, cost_usd DESC
LIMIT 50;


## Step 5 — How the dashboard wires this together

The companion **Scapia — Cost Attribution (User · Query · Table)** AI/BI dashboard has one dataset per query above, plus filters on workspace / date / product / catalog so users can flip between **per-workspace** and **account-wide** views.

| Page | Dataset | Visuals |
|------|---------|---------|
| 👤 Per-User | `per_user` (Step 2) | KPIs, top users, daily trend by product, breakdown table |
| 🔍 Per-Query | `per_query` (Step 3) | KPIs, top users, cost by statement_type, raw-query table |
| 📦 Per-Table / MV | `per_table` (Step 4) | KPIs, top tables, daily trend, cost by catalog, table breakdown |

## Step 6 — Reconciling with the standard *Account Usage Dashboard V2*

The walkthrough queries above use the **same price recipe** as the standard *Account Usage Dashboard V2*: `list_prices.pricing.effective_list.default`, joined on `sku_name + usage_unit`, time-bound to `usage_end_time`. That means the totals reconcile exactly.

Run this cell to confirm — the two rows should match on `total_usd`, `total_dbus`, and `row_count`.

In [ ]:
-- Total cost the standard Account Usage Dashboard V2 would show
-- with default parameters (price_table = list_prices, last 30d, no discounts)
WITH prices AS (
  SELECT sku_name, usage_unit, price_start_time,
         COALESCE(price_end_time, date_add(current_date, 1)) AS price_end_time,
         pricing.effective_list.default AS unit_price
  FROM system.billing.list_prices WHERE currency_code = 'USD'
),
dash1_totals AS (
  SELECT
    'dash1 (Account Usage Dashboard V2)'                     AS source,
    ROUND(SUM(u.usage_quantity * COALESCE(p.unit_price, 0)), 2) AS total_usd,
    ROUND(SUM(u.usage_quantity), 2)                             AS total_dbus,
    COUNT(*)                                                    AS row_count
  FROM system.billing.usage u
  LEFT JOIN prices p
    ON  u.sku_name   = p.sku_name
    AND u.usage_unit = p.usage_unit
    AND u.usage_end_time BETWEEN p.price_start_time AND p.price_end_time
  WHERE u.usage_date BETWEEN current_date() - 30 AND current_date()
),
dash2_totals AS (
  SELECT
    'dash2 (Scapia Cost Attribution)'                        AS source,
    ROUND(SUM(u.usage_quantity * COALESCE(p.unit_price, 0)), 2) AS total_usd,
    ROUND(SUM(u.usage_quantity), 2)                             AS total_dbus,
    COUNT(*)                                                    AS row_count
  FROM system.billing.usage u
  LEFT JOIN prices p
    ON  u.sku_name   = p.sku_name
    AND u.usage_unit = p.usage_unit
    AND u.usage_end_time BETWEEN p.price_start_time AND p.price_end_time
  WHERE u.usage_date >= current_date() - 30
)
SELECT * FROM dash1_totals
UNION ALL
SELECT * FROM dash2_totals;
